#### Tools Integration

#### Why Tools Matter in LangGraph

An LLM alone can reason and generate text — but it cannot fetch live data, run calculations, query a database, or call an external API. Tools are the bridge between the LLM's reasoning and the real world.

In LangGraph, tools are:

* Defined as Python functions decorated with @tool
* Bound to an LLM so it can decide when to call them
* Executed inside a ToolNode (a special graph node)
* Orchestrated by a ReAct agent loop (Reason → Act → Observe → Repeat)

##### Core Architecture — How Tool Calling Works in LangGraph

``` markdown
User Message
     ↓
  LLM Node  ──── decides to call tool ────→  ToolNode (executes tool)
     ↑                                              ↓
     └──────────── tool result (observation) ───────┘
     
  (loop continues until LLM decides no more tools needed)
     ↓
Final Response
```

#### Step by step:

1. LLM receives the user message
2. LLM returns a message with tool_calls (name + args) instead of plain text
3. ToolNode detects tool_calls, executes the matching Python function
4. Result is added to messages as a ToolMessage
5. LLM sees the result and either calls another tool or gives the final answer


In [17]:
# Setup
# pip install langgraph langchain-openai langchain-community
# pip install requests sqlalchemy duckduckgo-search

from langgraph.graph import StateGraph, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_groq import ChatGroq

import os
from dotenv import load_dotenv
load_dotenv()
model_name = os.getenv("groq_model_name")
print(model_name)
llm_groq = ChatGroq(
    model = 'llama-3.1-8b-instant',
    api_key = os.getenv("groq_api_key"),
    temperature = 0,
)
print(llm_groq)

llama-3.3-70b-versatile
metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.13'}} output_version=None profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True} client=<groq.resources.chat.completions.Completions object at 0x000001D4346ABC40> async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001D4346AAAD0> model_name='llama-3.1-8b-instant' temperature=1e-08 model_kwargs={} groq_api_key=SecretStr('**********') groq_api_base=None groq_proxy=None


MessagesState is a built-in state schema with a single messages key that auto-accumulates (uses operator.add reducer). It's the standard state for tool-calling agents.

Part 1 — Tool Calling Fundamentals


1.1 Defining a Tool



In [19]:
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city.
    
    Args:
        city: The name of the city to get weather for.
    """
    # Real implementation would call a weather API
    weather_data = {
        "bangalore": "28°C, Partly cloudy",
        "mumbai": "32°C, Humid",
        "delhi": "35°C, Sunny"
    }
    return weather_data.get(city.lower(), f"Weather data not available for {city}")

print(get_weather.name) 

get_weather


#### Critical rules for tool definitions:
1. The docstring is what the LLM reads to decide when to use the tool — write it clearly
2. Type hints are required — they define the JSON schema sent to the LLM
3. The return value must be a string (or JSON-serializable — LangChain handles conversion)

1.2 Inspecting a Tool

``` python
print(get_weather.name)          # "get_weather"
print(get_weather.description)   # the docstring
print(get_weather.args_schema.schema())  # JSON schema of args
```

1.3 Building the Minimal Tool-Calling Agent

In [20]:
from langgraph.graph import StateGraph, MessagesState, END
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

@tool
def get_weather(city: str) -> str:
    """Get current weather for a city."""
    return f"Weather in {city}: 28°C, Partly cloudy"

@tool
def get_population(city: str) -> str:
    """Get the population of a city."""
    populations = {"bangalore": "13 million", "mumbai": "20 million"}
    return populations.get(city.lower(), "Population data unavailable")

tools = [get_weather, get_population]
llm = ChatGroq(model=model_name)
llm_with_tools = llm.bind_tools(tools)   # ← key step: LLM now knows about tools

def llm_node(state: MessagesState) -> dict:
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

tool_node = ToolNode(tools)   # ← handles execution of all tools

builder = StateGraph(MessagesState)
builder.add_node("llm", llm_node)
builder.add_node("tools", tool_node)
builder.set_entry_point("llm")

# tools_condition: if LLM returned tool_calls → go to tools, else → END
builder.add_conditional_edges("llm", tools_condition)
builder.add_edge("tools", "llm")   # after tool runs, go back to LLM

graph = builder.compile()

# Run it
result = graph.invoke({
    "messages": [HumanMessage(content="What's the weather and population of Bangalore?")]
})

for msg in result["messages"]:
    print(f"{msg.__class__.__name__}: {msg.content}")

HumanMessage: What's the weather and population of Bangalore?
AIMessage: 
ToolMessage: Weather in Bangalore: 28°C, Partly cloudy
ToolMessage: 13 million
AIMessage: The current weather in Bangalore is 28°C with partly cloudy conditions. The population of Bangalore is approximately 13 million.


1.4 How tools_condition Works Internally

tools_condition is a pre-built routing function that does this:

``` python
def tools_condition(state: MessagesState) -> str:
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return END
```

we can write your own version if you need custom routing logic.



1.5 Tool with Multiple Arguments and Complex Return



In [22]:
@tool
def calculate_loan_emi(
    principal: float,
    annual_rate: float,
    tenure_months: int
) -> str:
    """Calculate monthly EMI for a loan.
    
    Args:
        principal: Loan amount in INR
        annual_rate: Annual interest rate as percentage (e.g., 8.5 for 8.5%)
        tenure_months: Loan tenure in months
    """
    monthly_rate = annual_rate / (12 * 100)
    emi = principal * monthly_rate * (1 + monthly_rate)**tenure_months / \
          ((1 + monthly_rate)**tenure_months - 1)
    total_payment = emi * tenure_months
    total_interest = total_payment - principal
    
    return (
        f"Loan EMI Calculation:\n"
        f"  Principal: ₹{principal:,.0f}\n"
        f"  Monthly EMI: ₹{emi:,.2f}\n"
        f"  Total Payment: ₹{total_payment:,.2f}\n"
        f"  Total Interest: ₹{total_interest:,.2f}"
    )

1.6 Forcing Specific Tool Use

Sometimes we want to force the LLM to always use a specific tool:

``` python
# Force the LLM to use a specific tool
llm_forced = llm.bind_tools(tools, tool_choice="get_weather")

# Or force it to use any tool (not free-text)
llm_any_tool = llm.bind_tools(tools, tool_choice="any")

# Default: LLM decides (tool_choice="auto")
llm_auto = llm.bind_tools(tools, tool_choice="auto")
```



Part 2 — Search Tools



2.1 DuckDuckGo Search (Free, No API Key)



In [25]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

@tool
def web_search(query: str) -> str:
    """Search the web for current information on any topic.
    
    Args:
        query: The search query to look up
    """
    try:
        result = search.run(query)
        return result
    except Exception as e:
        return f"Search failed: {str(e)}"

2.2 Tavily Search (Best for AI Agents — Returns Structured Results)

``` python
# pip install tavily-python
# Set TAVILY_API_KEY environment variable

from langchain_community.tools.tavily_search import TavilySearchResults

tavily_tool = TavilySearchResults(
    max_results=3,
    search_depth="advanced",
    include_answer=True,
    include_raw_content=False
)

# Use directly as a tool
tools = [tavily_tool]
llm_with_tools = llm.bind_tools(tools)
```


2.3 Custom Search Tool with Result Formatting



In [27]:
import requests

@tool
def news_search(topic: str, max_results: int = 3) -> str:
    """Search for recent news articles on a topic.
    
    Args:
        topic: The news topic to search for
        max_results: Number of results to return (default 3)
    """
    # Using NewsAPI (newsapi.org — free tier available)
    api_key = "your_newsapi_key"
    url = f"https://newsapi.org/v2/everything"
    params = {
        "q": topic,
        "sortBy": "publishedAt",
        "pageSize": max_results,
        "apiKey": api_key
    }
    
    try:
        response = requests.get(url, params=params, timeout=10)
        data = response.json()
        
        if data["status"] != "ok":
            return f"News search failed: {data.get('message', 'Unknown error')}"
        
        articles = data.get("articles", [])
        if not articles:
            return f"No news found for: {topic}"
        
        result_lines = [f"News results for '{topic}':"]
        for i, article in enumerate(articles, 1):
            result_lines.append(
                f"{i}. {article['title']} ({article['source']['name']}, "
                f"{article['publishedAt'][:10]})"
            )
        
        return "\n".join(result_lines)
        
    except requests.Timeout:
        return "News search timed out. Try again."
    except Exception as e:
        return f"News search error: {str(e)}"

2.4 Full Search Agent

In [30]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

@tool
def web_search(query: str) -> str:
    """Search the web for up-to-date information.
    
    Args:
        query: Search query string
    """
    return search_tool.run(query)

#
# llm = llm_groq
llm = ChatGroq(model="llama-3.1-8b-instant")  # No Compound AI
tools = [web_search]
llm_with_tools = llm.bind_tools(tools)

def llm_node(state: MessagesState):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

tool_node = ToolNode(tools)

builder = StateGraph(MessagesState)
builder.add_node("llm", llm_node)
builder.add_node("tools", tool_node)
builder.set_entry_point("llm")
builder.add_conditional_edges("llm", tools_condition)
builder.add_edge("tools", "llm")

search_agent = builder.compile()

result = search_agent.invoke({
    "messages": [HumanMessage(content="What is the current INR to USD exchange rate?")]
})
print(result["messages"][-1].content)

c:\Users\subramani.v\AppData\Local\Programs\Python\Python310\lib\site-packages\langchain_community\utilities\duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
c:\Users\subramani.v\AppData\Local\Programs\Python\Python310\lib\site-packages\langchain_community\utilities\duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


The function call for the current INR to USD exchange rate on 12 July 2026 was not successful.


Part 3 — Calculator Tools

In [31]:
# 3.1 Basic Math Tool
import math

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression safely.
    
    Args:
        expression: Math expression to evaluate. E.g. '2 + 2', '15% of 8000', 'sqrt(144)'
    """
    try:
        # Safe eval with only math functions
        allowed_names = {
            "sqrt": math.sqrt, "pow": math.pow, "abs": abs,
            "round": round, "ceil": math.ceil, "floor": math.floor,
            "log": math.log, "log10": math.log10, "pi": math.pi, "e": math.e
        }
        
        # Handle percentage expressions
        expression = expression.replace("%", "/100")
        
        result = eval(expression, {"__builtins__": {}}, allowed_names)
        return f"Result: {result}"
        
    except ZeroDivisionError:
        return "Error: Division by zero"
    except Exception as e:
        return f"Calculation error: {str(e)}"


In [32]:
# 3.2 Financial Calculator Tools

@tool
def compound_interest(principal: float, rate: float, time_years: float, n: int = 12) -> str:
    """Calculate compound interest.
    
    Args:
        principal: Initial investment amount in INR
        rate: Annual interest rate as percentage (e.g., 7.5 for 7.5%)
        time_years: Time period in years
        n: Compounding frequency per year (12=monthly, 4=quarterly, 1=annually)
    """
    r = rate / 100
    amount = principal * (1 + r/n) ** (n * time_years)
    interest = amount - principal
    
    return (
        f"Compound Interest Calculation:\n"
        f"  Principal: ₹{principal:,.2f}\n"
        f"  Rate: {rate}% per annum\n"
        f"  Time: {time_years} years\n"
        f"  Final Amount: ₹{amount:,.2f}\n"
        f"  Interest Earned: ₹{interest:,.2f}"
    )

@tool
def currency_converter(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert currency amounts using approximate rates.
    
    Args:
        amount: Amount to convert
        from_currency: Source currency code (USD, EUR, GBP, INR)
        to_currency: Target currency code (USD, EUR, GBP, INR)
    """
    # Approximate rates (in real system, fetch live rates from API)
    rates_to_usd = {"USD": 1.0, "EUR": 1.08, "GBP": 1.27, "INR": 0.012}
    rates_from_usd = {"USD": 1.0, "EUR": 0.93, "GBP": 0.79, "INR": 84.0}
    
    from_curr = from_currency.upper()
    to_curr = to_currency.upper()
    
    if from_curr not in rates_to_usd or to_curr not in rates_from_usd:
        return f"Unsupported currency. Supported: {list(rates_to_usd.keys())}"
    
    usd_amount = amount * rates_to_usd[from_curr]
    result = usd_amount * rates_from_usd[to_curr]
    
    return f"{amount} {from_curr} = {result:.2f} {to_curr} (approximate)"

Part 4 — Database Tools

4.1 SQLite Tool (Local Database)



In [ ]:
import sqlite3
from contextlib import contextmanager

# Setup test database
def setup_demo_db():
    conn = sqlite3.connect("demo.db")
    cursor = conn.cursor()
    cursor.executescript("""
        CREATE TABLE IF NOT EXISTS orders (
            id INTEGER PRIMARY KEY,
            customer TEXT,
            product TEXT,
            amount REAL,
            status TEXT,
            created_date TEXT
        );
        
        INSERT OR IGNORE INTO orders VALUES
            (1, 'Ravi Kumar', 'Laptop', 75000, 'delivered', '2024-01-15'),
            (2, 'Priya Sharma', 'Phone', 25000, 'pending', '2024-01-20'),
            (3, 'Amit Singh', 'Tablet', 35000, 'shipped', '2024-01-22'),
            (4, 'Ravi Kumar', 'Headphones', 8000, 'delivered', '2024-01-25'),
            (5, 'Sunita Patel', 'Laptop', 85000, 'pending', '2024-01-28');
    """)
    conn.commit()
    conn.close()

setup_demo_db()

@tool
def query_database(sql_query: str) -> str:
    """Execute a read-only SQL query on the orders database.
    
    Args:
        sql_query: A SELECT SQL query to execute. Only SELECT statements are allowed.
    
    Tables available:
        - orders (id, customer, product, amount, status, created_date)
    """
    # Safety: only allow SELECT
    query_upper = sql_query.strip().upper()
    if not query_upper.startswith("SELECT"):
        return "Error: Only SELECT queries are allowed."
    
    # Block dangerous keywords
    blocked = ["DROP", "DELETE", "UPDATE", "INSERT", "ALTER", "CREATE"]
    for keyword in blocked:
        if keyword in query_upper:
            return f"Error: '{keyword}' operations are not permitted."
    
    try:
        conn = sqlite3.connect("demo.db")
        conn.row_factory = sqlite3.Row
        cursor = conn.cursor()
        cursor.execute(sql_query)
        rows = cursor.fetchall()
        conn.close()
        
        if not rows:
            return "Query returned no results."
        
        # Format as readable table
        columns = rows[0].keys()
        result_lines = [" | ".join(columns)]
        result_lines.append("-" * 60)
        for row in rows:
            result_lines.append(" | ".join(str(row[col]) for col in columns))
        
        return f"Query results ({len(rows)} rows):\n" + "\n".join(result_lines)
        
    except sqlite3.Error as e:
        return f"Database error: {str(e)}"

@tool
def get_table_schema(table_name: str) -> str:
    """Get the schema (columns and types) of a database table.
    
    Args:
        table_name: Name of the table to inspect
    """
    try:
        conn = sqlite3.connect("demo.db")
        cursor = conn.cursor()
        cursor.execute(f"PRAGMA table_info({table_name})")
        columns = cursor.fetchall()
        conn.close()
        
        if not columns:
            return f"Table '{table_name}' not found."
        
        schema_lines = [f"Schema for table '{table_name}':"]
        for col in columns:
            schema_lines.append(f"  {col[1]} ({col[2]})")
        
        return "\n".join(schema_lines)
        
    except sqlite3.Error as e:
        return f"Schema error: {str(e)}"

In [ ]:
# 4.2 Database Agent — Natural Language to SQL

# System prompt for the SQL agent
SQL_SYSTEM_PROMPT = """You are a SQL assistant with access to an orders database.

When users ask questions about orders, customers, or products:
1. First use get_table_schema to understand the table structure if needed
2. Then use query_database to run appropriate SELECT queries
3. Interpret results clearly and answer the user's question

Always verify your query syntax before running. Only use SELECT statements."""

from langchain_core.messages import SystemMessage

tools = [query_database, get_table_schema]
llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tools = llm.bind_tools(tools)

def llm_node(state: MessagesState):
    messages = [SystemMessage(content=SQL_SYSTEM_PROMPT)] + state["messages"]
    return {"messages": [llm_with_tools.invoke(messages)]}

tool_node = ToolNode(tools)

builder = StateGraph(MessagesState)
builder.add_node("llm", llm_node)
builder.add_node("tools", tool_node)
builder.set_entry_point("llm")
builder.add_conditional_edges("llm", tools_condition)
builder.add_edge("tools", "llm")

db_agent = builder.compile()

# Test queries
queries = [
    "How many pending orders do we have?",
    "What is the total revenue from delivered orders?",
    "Which customer has spent the most?"
]

for query in queries:
    print(f"\nQ: {query}")
    result = db_agent.invoke({"messages": [HumanMessage(content=query)]})
    print(f"A: {result['messages'][-1].content}")

Part 5 — API Tools

5.1 REST API Tool (Generic Pattern)



In [33]:
import requests
from typing import Optional

@tool
def call_rest_api(
    url: str,
    method: str = "GET",
    params: Optional[dict] = None,
    headers: Optional[dict] = None
) -> str:
    """Make a REST API call and return the response.
    
    Args:
        url: The API endpoint URL
        method: HTTP method (GET, POST)
        params: Query parameters as a dict
        headers: Request headers as a dict
    """
    try:
        if method.upper() == "GET":
            response = requests.get(url, params=params, headers=headers, timeout=10)
        elif method.upper() == "POST":
            response = requests.post(url, json=params, headers=headers, timeout=10)
        else:
            return f"Unsupported method: {method}"
        
        response.raise_for_status()
        
        # Try to return JSON, fall back to text
        try:
            data = response.json()
            # Truncate if too long
            import json
            text = json.dumps(data, indent=2)
            return text[:2000] + "..." if len(text) > 2000 else text
        except:
            return response.text[:2000]
            
    except requests.Timeout:
        return "API call timed out."
    except requests.HTTPError as e:
        return f"HTTP error: {e.response.status_code} — {e.response.text[:200]}"
    except Exception as e:
        return f"API error: {str(e)}"

In [34]:
# 5.2 Specific API Tools — Weather API

import os

@tool
def get_live_weather(city: str) -> str:
    """Get live weather data for a city using OpenWeatherMap API.
    
    Args:
        city: City name (e.g., 'Bangalore', 'Mumbai')
    """
    api_key = os.getenv("OPENWEATHER_API_KEY")
    if not api_key:
        return "Weather API key not configured."
    
    url = "https://api.openweathermap.org/data/2.5/weather"
    params = {"q": city, "appid": api_key, "units": "metric"}
    
    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        return (
            f"Weather in {data['name']}, {data['sys']['country']}:\n"
            f"  Temperature: {data['main']['temp']}°C "
            f"(feels like {data['main']['feels_like']}°C)\n"
            f"  Condition: {data['weather'][0]['description'].title()}\n"
            f"  Humidity: {data['main']['humidity']}%\n"
            f"  Wind: {data['wind']['speed']} m/s"
        )
    except requests.HTTPError as e:
        if e.response.status_code == 404:
            return f"City '{city}' not found."
        return f"Weather API error: {e}"
    except Exception as e:
        return f"Error: {str(e)}"

In [35]:
# 5.3 GitHub API Tool
@tool
def get_github_repo_info(owner: str, repo: str) -> str:
    """Get information about a GitHub repository.
    
    Args:
        owner: GitHub username or organization name
        repo: Repository name
    """
    url = f"https://api.github.com/repos/{owner}/{repo}"
    headers = {"Accept": "application/vnd.github.v3+json"}
    
    # Add token if available
    token = os.getenv("GITHUB_TOKEN")
    if token:
        headers["Authorization"] = f"token {token}"
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        return (
            f"Repository: {data['full_name']}\n"
            f"  Description: {data.get('description', 'No description')}\n"
            f"  Stars: {data['stargazers_count']:,}\n"
            f"  Forks: {data['forks_count']:,}\n"
            f"  Open Issues: {data['open_issues_count']}\n"
            f"  Primary Language: {data.get('language', 'Unknown')}\n"
            f"  Last Updated: {data['updated_at'][:10]}\n"
            f"  URL: {data['html_url']}"
        )
    except requests.HTTPError as e:
        if e.response.status_code == 404:
            return f"Repository '{owner}/{repo}' not found."
        return f"GitHub API error: {str(e)}"



In [36]:
# 5.4 Rate-Limited API Tool (Production Pattern)
import time
from functools import wraps

# Simple in-memory rate limiter
_last_call_times = {}

def rate_limit(calls_per_minute: int):
    """Decorator to rate-limit API calls."""
    min_interval = 60.0 / calls_per_minute
    
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            func_name = func.__name__
            last_time = _last_call_times.get(func_name, 0)
            elapsed = time.time() - last_time
            
            if elapsed < min_interval:
                time.sleep(min_interval - elapsed)
            
            _last_call_times[func_name] = time.time()
            return func(*args, **kwargs)
        return wrapper
    return decorator

@tool
@rate_limit(calls_per_minute=30)
def search_product_catalog(query: str, category: str = "") -> str:
    """Search product catalog with rate limiting.
    
    Args:
        query: Product search query
        category: Optional product category filter
    """
    # Simulated product catalog search
    products = [
        {"name": "Laptop Pro X1", "price": 75000, "category": "electronics"},
        {"name": "Wireless Headphones", "price": 8000, "category": "electronics"},
        {"name": "Office Chair", "price": 15000, "category": "furniture"},
    ]
    
    results = [
        p for p in products
        if query.lower() in p["name"].lower()
        and (not category or category.lower() == p["category"])
    ]
    
    if not results:
        return f"No products found for '{query}'"
    
    lines = [f"Products matching '{query}':"]
    for p in results:
        lines.append(f"  {p['name']} — ₹{p['price']:,} ({p['category']})")
    
    return "\n".join(lines)


Part 6 — External Services



6.1 Email Service Tool (SendGrid Pattern)



In [37]:
@tool
def send_email_notification(
    to_email: str,
    subject: str,
    body: str
) -> str:
    """Send an email notification to a user.
    
    Args:
        to_email: Recipient email address
        subject: Email subject line
        body: Email body content (plain text)
    """
    # In production: use sendgrid, SES, or SMTP
    # import sendgrid
    # sg = sendgrid.SendGridAPIClient(api_key=os.getenv("SENDGRID_API_KEY"))
    
    # Simulation
    print(f"[EMAIL SERVICE] Sending to: {to_email}")
    print(f"  Subject: {subject}")
    print(f"  Body: {body[:100]}...")
    
    return f"Email sent successfully to {to_email} with subject '{subject}'"

In [38]:
# 6.2 Slack Notification Tool
@tool
def send_slack_message(channel: str, message: str) -> str:
    """Send a message to a Slack channel.
    
    Args:
        channel: Slack channel name (e.g., '#alerts', '#general')
        message: Message text to send
    """
    slack_token = os.getenv("SLACK_BOT_TOKEN")
    
    if not slack_token:
        # Simulation mode
        print(f"[SLACK] → {channel}: {message}")
        return f"Message sent to {channel}"
    
    url = "https://slack.com/api/chat.postMessage"
    headers = {"Authorization": f"Bearer {slack_token}"}
    payload = {"channel": channel, "text": message}
    
    try:
        response = requests.post(url, json=payload, headers=headers, timeout=10)
        data = response.json()
        
        if data.get("ok"):
            return f"Message sent to {channel}"
        else:
            return f"Slack error: {data.get('error', 'Unknown error')}"
    except Exception as e:
        return f"Slack service error: {str(e)}"


In [39]:
# 6.3 File System Tool

import os
from pathlib import Path

ALLOWED_BASE_DIR = "/tmp/agent_workspace"  # Restrict to safe directory
os.makedirs(ALLOWED_BASE_DIR, exist_ok=True)

@tool
def read_file(filename: str) -> str:
    """Read content from a file in the workspace.
    
    Args:
        filename: Name of the file to read (no path traversal allowed)
    """
    # Security: prevent path traversal
    safe_path = Path(ALLOWED_BASE_DIR) / Path(filename).name
    
    if not safe_path.exists():
        return f"File '{filename}' not found in workspace."
    
    try:
        with open(safe_path, "r") as f:
            content = f.read()
        return f"Content of {filename}:\n{content[:3000]}"
    except Exception as e:
        return f"File read error: {str(e)}"

@tool
def write_file(filename: str, content: str) -> str:
    """Write content to a file in the workspace.
    
    Args:
        filename: Name of the file to create/overwrite
        content: Text content to write
    """
    safe_path = Path(ALLOWED_BASE_DIR) / Path(filename).name
    
    try:
        with open(safe_path, "w") as f:
            f.write(content)
        return f"File '{filename}' written successfully ({len(content)} characters)"
    except Exception as e:
        return f"File write error: {str(e)}"

Part 7 — Full Production Agent (All Tool Types Combined)

This is the capstone — a multi-tool business assistant that combines search, calculator, database, and notification tools in one agent.



In [41]:
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, MessagesState, END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

# --- Define all tools ---

@tool
def web_search(query: str) -> str:
    """Search the web for current information."""
    from langchain_community.tools import DuckDuckGoSearchRun
    return DuckDuckGoSearchRun().run(query)

@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression. E.g. '1500 * 0.18' for GST calculation."""
    import math
    allowed = {"sqrt": math.sqrt, "pow": pow, "abs": abs, "round": round, "pi": math.pi}
    try:
        return str(eval(expression, {"__builtins__": {}}, allowed))
    except Exception as e:
        return f"Calc error: {e}"

@tool
def query_orders_db(sql: str) -> str:
    """Query the orders database. Only SELECT statements allowed."""
    import sqlite3
    if not sql.strip().upper().startswith("SELECT"):
        return "Only SELECT queries allowed."
    try:
        conn = sqlite3.connect("demo.db")
        conn.row_factory = sqlite3.Row
        cur = conn.cursor()
        cur.execute(sql)
        rows = cur.fetchall()
        conn.close()
        if not rows:
            return "No results."
        cols = rows[0].keys()
        lines = [" | ".join(cols), "-"*50]
        lines += [" | ".join(str(r[c]) for c in cols) for r in rows]
        return "\n".join(lines)
    except Exception as e:
        return f"DB error: {e}"

@tool
def send_notification(recipient: str, message: str) -> str:
    """Send a notification to a team member or customer.
    Args:
        recipient: Name or email of recipient
        message: Notification message
    """
    print(f"\n[NOTIFICATION → {recipient}]: {message}")
    return f"Notification sent to {recipient}"

# --- Build the agent ---

SYSTEM_PROMPT = """You are a business intelligence assistant for an e-commerce company.

You have access to:
- web_search: for market research and external information
- calculate: for math, percentages, financial calculations
- query_orders_db: to query the orders table (columns: id, customer, product, amount, status, created_date)
- send_notification: to alert team members

Always show your reasoning. When querying the database, start with the right SQL query.
For financial calculations, show the breakdown clearly."""

tools = [web_search, calculate, query_orders_db, send_notification]
llm = ChatGroq(model=model_name)
llm_with_tools = llm.bind_tools(tools)

def llm_node(state: MessagesState):
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    return {"messages": [llm_with_tools.invoke(messages)]}

tool_node = ToolNode(tools)

builder = StateGraph(MessagesState)
builder.add_node("llm", llm_node)
builder.add_node("tools", tool_node)
builder.set_entry_point("llm")
builder.add_conditional_edges("llm", tools_condition)
builder.add_edge("tools", "llm")

agent = builder.compile(checkpointer=MemorySaver())

# --- Test scenarios ---

test_cases = [
    "What is the total revenue from all delivered orders? Also calculate the GST (18%) on that amount.",
    "How many pending orders are there and what is their combined value?",
    "Find all orders by Ravi Kumar and notify the sales team about his total spend.",
]

config = {"configurable": {"thread_id": "biz-agent-001"}}

for question in test_cases:
    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print("="*60)
    result = agent.invoke({"messages": [HumanMessage(content=question)]}, config)
    print(f"A: {result['messages'][-1].content}")


Q: What is the total revenue from all delivered orders? Also calculate the GST (18%) on that amount.
A: Assuming the total revenue from all delivered orders is $1000.

The total revenue from all delivered orders is $1000. 
To calculate the GST (18%) on this amount: 
$1000 * 0.18 = $180 
So, the GST on the total revenue is $180.

Q: How many pending orders are there and what is their combined value?
A: Let's try to break it down:

To find the number of pending orders and their combined value, we need to query the orders database. 

First, we'll count the number of rows in the orders table where the status is 'pending'. This will give us the number of pending orders.

Then, we'll sum up the 'amount' column for the same rows. This will give us the combined value of the pending orders.

We can achieve this with a single SQL query that uses both the COUNT and SUM functions. 

However, since the previous attempt resulted in a "no such table" error, let's try to assume some numbers for the s

Part 8 — Error Handling in Tools

8.1 The ToolException Pattern


In [42]:
from langchain_core.tools import ToolException

@tool
def fetch_customer_data(customer_id: str) -> str:
    """Fetch customer data by ID.
    
    Args:
        customer_id: The unique customer identifier
    """
    if not customer_id.startswith("CUST-"):
        raise ToolException(
            f"Invalid customer ID format: '{customer_id}'. "
            f"Expected format: CUST-XXXXX"
        )
    
    # Simulate lookup
    customers = {"CUST-001": "Ravi Kumar", "CUST-002": "Priya Sharma"}
    customer = customers.get(customer_id)
    
    if not customer:
        raise ToolException(f"Customer '{customer_id}' not found in database.")
    
    return f"Customer: {customer} (ID: {customer_id})"

# ToolNode automatically handles ToolException
# It converts it to a ToolMessage with the error text
# The LLM then sees the error and can retry or inform the user

In [43]:
# 8.2 Graceful Fallback Pattern
@tool
def get_stock_price(ticker: str) -> str:
    """Get current stock price for a ticker symbol.
    
    Args:
        ticker: Stock ticker symbol (e.g., RELIANCE, TCS, INFY)
    """
    try:
        # Try primary source
        price = _fetch_from_primary_api(ticker)
        return f"{ticker}: ₹{price:,.2f}"
    except Exception as primary_error:
        try:
            # Fallback to secondary source
            price = _fetch_from_fallback_api(ticker)
            return f"{ticker}: ₹{price:,.2f} (from fallback source)"
        except Exception:
            return (
                f"Could not fetch price for {ticker}. "
                f"Please check the ticker symbol and try again."
            )

def _fetch_from_primary_api(ticker): raise Exception("Primary down")  # Simulated
def _fetch_from_fallback_api(ticker): return 2850.50  # Simulated fallback


Part 9 — Tool Best Practices

9.1 Tool Naming and Docstrings

The LLM decides which tool to call based entirely on the name and docstring. Write them as if instructing a smart colleague.

``` python
# BAD — vague name, poor docstring
@tool
def process(x: str) -> str:
    """Process the input."""
    ...

# GOOD — clear name, descriptive docstring
@tool
def calculate_gst_amount(base_amount: float, gst_rate: float = 18.0) -> str:
    """Calculate GST (Goods and Services Tax) for a given base amount.
    
    Args:
        base_amount: The pre-tax amount in INR
        gst_rate: GST percentage rate (default 18%). Common rates: 5, 12, 18, 28
    
    Returns the GST amount and total including GST.
    """
    ...
```

9.2 Limit Tool Scope

Each tool should do exactly one thing. Don't build a god-tool.

``` python
# BAD — does too many things
@tool
def manage_order(action: str, order_id: str, new_status: str = "") -> str:
    """Get, update, or cancel an order."""
    ...

# GOOD — one responsibility each
@tool
def get_order_details(order_id: str) -> str:
    """Get full details of a specific order by ID."""
    ...

@tool
def update_order_status(order_id: str, new_status: str) -> str:
    """Update the status of an order."""
    ...
```

9.3 Always Validate Inputs

``` python
@tool
def get_date_range_orders(start_date: str, end_date: str) -> str:
    """Get orders within a date range.
    
    Args:
        start_date: Start date in YYYY-MM-DD format
        end_date: End date in YYYY-MM-DD format
    """
    from datetime import datetime
    
    try:
        start = datetime.strptime(start_date, "%Y-%m-%d")
        end = datetime.strptime(end_date, "%Y-%m-%d")
    except ValueError:
        return "Invalid date format. Use YYYY-MM-DD (e.g., 2024-01-15)"
    
    if end < start:
        return "end_date must be after start_date"
    
    # Proceed with validated dates
    ...
```